# Experiment 01 — Baseline TFNO2D (Kaggle First Run)

This notebook is the **first official baseline run** using the competition's 16 input features only.
All reusable code lives in `src/`.

**Notes:**
- Uses official min-max normalization (`feat_min_max.mat`)
- Uses strict temporal blocking (`OCT_16` validation)
- Uses baseline `tfno2d` model (no extra auxiliary channels)
- Uses persistence gate from EDA to verify learning quality

**Pipeline:**  
1. Load config  
2. Load official normalization bounds  
3. Build train/validation loaders  
4. Train baseline TFNO2D and compare against persistence  
5. Run inference and save `preds.npy`

In [ ]:
import sys, os

# ── Path resolution: works both locally and on Kaggle ──
# On Kaggle: attach your src dataset as input, set KAGGLE_SRC_DATASET below.
# Locally:   runs from notebooks/ so '../' resolves to Ronit/

KAGGLE_SRC_DATASET = "ronit-pm25-src"

LOCAL_SRC  = os.path.abspath('../')
KAGGLE_SRC = f'/kaggle/input/datasets/kushieboi/{KAGGLE_SRC_DATASET}'

if os.path.exists('/kaggle'):
    SRC_ROOT = KAGGLE_SRC
else:
    SRC_ROOT = LOCAL_SRC

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")
from src.config import load_config
from src.utils import seed_everything, print_device_info, count_parameters, sanity_check_bounds
from src.data import load_minmax_bounds, load_all_months, get_dataloaders
from src.model import build_model
from src.train import train
from src.inference import run_inference

## 1. Configuration

In [ ]:
cfg = load_config()

seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

print(f"Base features ({cfg['features']['n_features']}): {cfg['features']['base']}")
print(f"Aux features  ({len(cfg['features']['aux'])}): {cfg['features']['aux']}")
print(f"Input channels: {cfg['features']['input_channels']}")
print(f"Train months: {cfg['data']['train_months']}")
print(f"Val month:    {cfg['data']['val_month']}")
print(f"Model type:   {cfg['model']['type']}")
print(f"Data root:    {cfg['paths']['data']}")

## 2. Load Normalization Bounds (feat_min_max.mat)

In [ ]:
# Load official min-max bounds from feat_min_max.mat
bounds = load_minmax_bounds(cfg)
print(f"Loaded bounds for {len(bounds)} features")
print(f"Source: {cfg['paths']['min_max']}")

# Sanity check: print ranges, flag any zero-range features
sanity_check_bounds(bounds, cfg['features']['all'])

## 3. Load & Preprocess Training / Validation Data

Baseline preprocessing used here:
- official `feat_min_max.mat` normalization for all 16 competition features
- strict temporal blocking (`OCT_16` held out entirely)
- lazy sliding-window dataset to avoid materializing massive tensors

In [ ]:
print("Loading + normalizing training months ...")
train_data = load_all_months(cfg, cfg['data']['train_months'], bounds)

print("\nLoading + normalizing validation month ...")
val_data = load_all_months(cfg, [cfg['data']['val_month']], bounds)

In [ ]:
train_dl, val_dl = get_dataloaders(cfg, train_data, val_data, bounds)
print("Batch shape check ...")
xb, yb = next(iter(train_dl))
print(f"  x: {tuple(xb.shape)}  (B, C={xb.shape[1]}, T={xb.shape[2]}, H={xb.shape[3]}, W={xb.shape[4]})")
print(f"  y: {tuple(yb.shape)}  (B, H, W, T_out={yb.shape[3]})")
print(f"  x range: [{xb.min():.3f}, {xb.max():.3f}]")
print(f"  y range: [{yb.min():.3f}, {yb.max():.3f}]")
print("  Baseline mode: only 16 official features (no auxiliary channels)")

## 4. Build & Train Baseline TFNO2D

In [ ]:
model = build_model(cfg)
print(f"Parameters: {count_parameters(model):,}")
print(f"Using model: {cfg['model']['type']}")
print("Expected first target: beat persistence gate before trying larger architectures")

In [ ]:
history = train(cfg, model, train_dl, val_dl, bounds=bounds)

### Training Curves

In [ ]:
import matplotlib.pyplot as plt
from src.train import PERSISTENCE_RMSE_NORM

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Left: normalized RMSE ──
ax = axes[0]
ax.plot(history['train_loss'], label='Train RMSE (norm)', alpha=0.8)
ax.plot(history['val_loss'],   label='Val RMSE (norm)',   alpha=0.9)
if 'val_persistence' in history:
    ax.plot(history['val_persistence'], label='Val persistence RMSE', alpha=0.8)
ax.axhline(PERSISTENCE_RMSE_NORM, color='red', linestyle='--',
           label=f'Global persistence gate ({PERSISTENCE_RMSE_NORM})')
ax.set_xlabel('Epoch');  ax.set_ylabel('Normalized RMSE')
ax.set_title('Training Curves — Normalized Space')
ax.legend();  ax.grid(True, alpha=0.3)

# ── Right: physical RMSE estimate ──
ax = axes[1]
fmin, fmax = bounds['cpm25']
cpm_range  = fmax - fmin
ax.plot([v * cpm_range for v in history['train_loss']], label='Train RMSE (µg/m³)', alpha=0.8)
ax.plot([v * cpm_range for v in history['val_loss']],   label='Val RMSE (µg/m³)',   alpha=0.9)
if 'val_persistence' in history:
    ax.plot([v * cpm_range for v in history['val_persistence']], label='Val persistence (µg/m³)', alpha=0.8)
ax.axhline(30.83, color='red', linestyle='--', label='Global persistence baseline (30.83 µg/m³)')
ax.set_xlabel('Epoch');  ax.set_ylabel('RMSE (µg/m³)')
ax.set_title('Training Curves — Physical Units (estimate)')
ax.legend();  ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Inference & Submit

In [ ]:
import torch

model.load_state_dict(torch.load(cfg['paths']['model_save'], map_location=cfg['device']))
preds = run_inference(cfg, model, bounds)
print('Done!')